# Generate Kihu vs AdvancedPlayer

`MyPlayer` を `AdvancedPlayer` と合計10局対戦させ、学習用の1行1局棋譜を専用フォルダへ保存します。序盤だけランダム合法手を混ぜて、同じ棋譜ばかりになるのを避けます。

In [ ]:
from pathlib import Path
import json
import os
import random
import time

from othellopy.game import OthelloGame
from othellopy.players import AdvancedPlayer, BasePlayer, BeginnerPlayer, IntermediatePlayer
from othellopy.core import Board, Cell, Move


MYPLAYER_FILE = os.environ.get("MYPLAYER_FILE", "players/current.py")
OPPONENT_PLAYER = os.environ.get("OPPONENT_PLAYER", "AdvancedPlayer")
TOTAL_GAMES = int(os.environ.get("TOTAL_GAMES", "10"))
RANDOM_OPENING_PLIES = int(os.environ.get("RANDOM_OPENING_PLIES", "8"))
RANDOM_SEED = int(os.environ.get("RANDOM_SEED", "42"))
MOVE_TIMEOUT_SECONDS_TEXT = os.environ.get("MOVE_TIMEOUT_SECONDS", "none")
MOVE_TIMEOUT_SECONDS = None if MOVE_TIMEOUT_SECONDS_TEXT.lower() in ("", "none") else float(MOVE_TIMEOUT_SECONDS_TEXT)

DEFAULT_OUTPUT_ROOT = f"generated_kihu/for_training/vs_{Path(OPPONENT_PLAYER).stem}_10games"
OUTPUT_ROOT = Path(os.environ.get("KIHU_OUTPUT_ROOT", DEFAULT_OUTPUT_ROOT))
RUN_NAME = os.environ.get("KIHU_RUN_NAME", time.strftime("%Y%m%d_%H%M%S"))
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("player:", MYPLAYER_FILE)
print("opponent:", OPPONENT_PLAYER)
print("games:", TOTAL_GAMES)
print("random opening plies:", RANDOM_OPENING_PLIES)
print("output dir:", OUTPUT_DIR)


In [ ]:
def _resolve_player_path(player_file):
    candidates = [
        Path(player_file),
        Path.cwd() / player_file,
        Path.cwd() / "players" / player_file,
        Path.cwd() / "players/baselines" / player_file,
        Path.cwd() / "players/experiments" / player_file,
        Path.cwd() / "players/random_opening" / player_file,
        Path.cwd() / "players" / Path(player_file).name,
        Path.cwd() / "players/baselines" / Path(player_file).name,
        Path.cwd() / "players/experiments" / Path(player_file).name,
        Path.cwd() / "players/random_opening" / Path(player_file).name,
        Path.cwd() / "my_othello_ai/pattern_eval/players" / Path(player_file).name,
        Path.cwd() / "my_othello_ai/pattern_eval/players/baselines" / Path(player_file).name,
        Path.cwd() / "my_othello_ai/pattern_eval/players/experiments" / Path(player_file).name,
        Path.cwd() / "my_othello_ai/pattern_eval/players/random_opening" / Path(player_file).name,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Player file not found: {player_file}")


def _load_player_from_file(player_file, class_name="MyPlayer"):
    path = _resolve_player_path(player_file)
    namespace = {"BasePlayer": BasePlayer, "Board": Board, "Cell": Cell, "Move": Move}
    exec(path.read_text(encoding="utf-8"), namespace)
    return namespace[class_name], path


def _load_myplayer(player_file):
    return _load_player_from_file(player_file)


def _load_opponent(player_spec):
    builtins = {
        "AdvancedPlayer": AdvancedPlayer,
        "Advanced": AdvancedPlayer,
        "IntermediatePlayer": IntermediatePlayer,
        "Intermediate": IntermediatePlayer,
        "BeginnerPlayer": BeginnerPlayer,
        "Beginner": BeginnerPlayer,
    }
    if player_spec in builtins:
        return builtins[player_spec], player_spec
    player_cls, path = _load_player_from_file(player_spec)
    return player_cls, str(path)


MyPlayer, MYPLAYER_PATH = _load_myplayer(MYPLAYER_FILE)
OpponentPlayer, OPPONENT_NAME = _load_opponent(OPPONENT_PLAYER)
print("loaded:", MYPLAYER_PATH)
print("loaded opponent:", OPPONENT_NAME)


In [ ]:
def _player_name(player):
    return getattr(player, "__name__", player.__class__.__name__)


def _move_to_record_coord(move):
    if move is None:
        return ""
    row, col = move
    return chr(ord("a") + col) + str(row + 1)


def _get_attr(obj, names, default=None):
    for name in names:
        if isinstance(obj, dict) and name in obj:
            return obj[name]
        if hasattr(obj, name):
            value = getattr(obj, name)
            return value() if callable(value) else value
    return default


def _record_text_from_result(result):
    turns = _get_attr(result, ("turns",), []) or []
    coords = []
    for turn in turns:
        coord = _move_to_record_coord(_get_attr(turn, ("move",), None))
        if coord:
            coords.append(coord)
    return "".join(coords)


def _skip_reason_from_result(result, record_text):
    forfeit = _get_attr(result, ("forfeit",), None)
    if forfeit is not None:
        message = _get_attr(forfeit, ("message",), "forfeit")
        return str(message).splitlines()[0]
    if not record_text:
        return "empty record_text"
    return ""


def _count_stones_from_board(board):
    black = white = 0
    for row in board:
        for cell in row:
            value = getattr(cell, "value", cell)
            if value == 1:
                black += 1
            elif value == 2:
                white += 1
    return black, white


def _result_summary(result, game=None):
    board = _get_attr(result, ("board", "final_board"), None)
    black_score = _get_attr(result, ("black_score", "black_count", "black"), None)
    white_score = _get_attr(result, ("white_score", "white_count", "white"), None)
    if game is not None:
        board = board or _get_attr(game, ("board", "final_board"), None)
    if (black_score is None or white_score is None) and board is not None:
        black_score, white_score = _count_stones_from_board(board)
    black_score = int(black_score)
    white_score = int(white_score)
    if black_score > white_score:
        winner = "black"
    elif white_score > black_score:
        winner = "white"
    else:
        winner = "draw"
    return {"black_score": black_score, "white_score": white_score, "winner": winner}


def _make_player_class(player_cls, label, game_index, move_logs, random_opening=False, turn_state=None, rng=None):
    class GeneratedPlayer:
        def __init__(self, *args, **kwargs):
            self._inner = player_cls(*args, **kwargs) if isinstance(player_cls, type) else player_cls

        def __getattr__(self, name):
            return getattr(self._inner, name)

        def next_move(self, board):
            start = time.perf_counter()
            randomized = False
            if random_opening and turn_state is not None and turn_state["plies"] < RANDOM_OPENING_PLIES:
                moves = list(self._inner.get_moves(board))
                if moves:
                    move = rng.choice(moves)
                    randomized = True
                else:
                    move = self._inner.next_move(board)
            else:
                move = self._inner.next_move(board)
            if turn_state is not None:
                turn_state["plies"] += 1
            elapsed = time.perf_counter() - start
            move_logs.append({
                "game_index": game_index,
                "player": label,
                "move": _move_to_record_coord(move),
                "elapsed_ms": elapsed * 1000,
                "randomized": randomized,
            })
            return move

    GeneratedPlayer.__name__ = f"Generated{label}"
    return GeneratedPlayer

In [ ]:
records = []
game_summaries = []

for game_index in range(1, TOTAL_GAMES + 1):
    my_is_black = game_index % 2 == 1
    move_logs = []
    turn_state = {"plies": 0}
    rng = random.Random(RANDOM_SEED + game_index)
    my_player = _make_player_class(
        MyPlayer,
        "MyPlayer",
        game_index,
        move_logs,
        random_opening=True,
        turn_state=turn_state,
        rng=rng,
    )
    opponent_player = _make_player_class(OpponentPlayer, Path(str(OPPONENT_NAME)).stem, game_index, move_logs)

    if my_is_black:
        black_player, white_player = my_player, opponent_player
        my_color = "black"
    else:
        black_player, white_player = opponent_player, my_player
        my_color = "white"

    game = OthelloGame(
        black_player=black_player,
        white_player=white_player,
        move_timeout_seconds=MOVE_TIMEOUT_SECONDS,
    )
    result = game.play()
    record_text = _record_text_from_result(result)
    summary = _result_summary(result, game)
    skip_reason = _skip_reason_from_result(result, record_text)
    summary.update({
        "game_index": game_index,
        "my_color": my_color,
        "record_text": record_text,
        "completed": not skip_reason,
        "skip_reason": skip_reason,
        "randomized_moves": sum(1 for log in move_logs if log["randomized"]),
    })
    records.append(record_text)
    game_summaries.append(summary)
    status = "ok" if not skip_reason else f"skip={skip_reason}"
    print(
        f"game {game_index:02d}: my={my_color} "
        f"black={summary['black_score']} white={summary['white_score']} "
        f"winner={summary['winner']} moves={len(record_text) // 2} {status}"
    )


In [ ]:
record_path = OUTPUT_DIR / "records.txt"
summary_path = OUTPUT_DIR / "summary.json"

record_lines = [game["record_text"] for game in game_summaries if game["completed"] and game["record_text"]]
record_path.write_text("\n".join(record_lines) + ("\n" if record_lines else ""), encoding="utf-8")

summary = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "myplayer_file": str(MYPLAYER_PATH),
    "opponent": str(OPPONENT_NAME),
    "total_games": TOTAL_GAMES,
    "saved_games": len(record_lines),
    "random_opening_plies": RANDOM_OPENING_PLIES,
    "random_seed": RANDOM_SEED,
    "move_timeout_seconds": MOVE_TIMEOUT_SECONDS,
    "record_path": str(record_path),
    "games": game_summaries,
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("records:", record_path)
print("summary:", summary_path)
